Combining the two datatsets (arxiv and acl) as well as removing the papers that couldn't be collected from the Dataframe

In [ ]:
#df = pandas.read_excel("journal_with_filtered_titles.xlsx")
#df_acl = pandas.read_excel("ACL_Anthology/acl_filtered_title.xlsx")

In [ ]:
liste = os.listdir("Texts")

new_list = []
mini_new_list = []

for file in liste:
    name = "10.48550/" + str(file[9:-3]) #aus dem String wird die Zahlenkombi, die alle Teilen rausgenommen und die Dateiendung entfernt
    dic = {}
    row = df.loc[df["doi"]==name].to_dict('records')[0]
    #print(row["doi"])
    new_list.append(row)
    dic["doi"] = row["doi"]
    dic["title"] = row["title"]
    dic["author"] = row["authors"]
    dic["filtered_titles"] = row["filtered_titles"]
    mini_new_list.append(dic)


In [ ]:
df_reduziert = pandas.DataFrame(new_list, columns=["Unnamed: 0", "doi", "date", "title", "authors", "abstract", "journal", "filtered_titles", "List_entries"])
mini_df = pandas.DataFrame(mini_new_list, columns=["doi", "title", "author","filtered_titles"])

In [ ]:
df_reduziert.rename({"Unnamed: 0": "prev Index"}, inplace=True)
df_reduziert.to_excel("Metadata_Texte.xlsx")

In [ ]:
mini_combi_df = pandas.concat([mini_df, df_acl], axis=0, ignore_index=True) 
mini_combi_df.to_excel("mini_df_plus_acl.xlsx")

In [ ]:
df_meta.journal.str.count("ACM").sum()

As the human papers still contain information about their authors and institutions, this skript removes these informations from the markdown files and also simplifies the contained figures to make it harder to used that as detection feature.

In [ ]:
import pandas as pd
import os
import re
import random

liste = os.listdir("Human_Corpus")

#the following two variables as well as the three files thereafter are just for control and status information
counter = 0
failure = []
key = {} #dictionary to save the key with which the papers are anonymized so they can be traced back

log = open("log.txt", "a+", encoding="utf-8")
strings = open("strings_to_replace.txt", "a+", encoding="utf-8")
listen = open("listen.txt", "a+", encoding="utf-8")

for filename in liste:
    path = "Human_Corpus/" + filename

    try:
        with open(path, "r+", encoding="utf-8") as file: #every file in the directory are gone through and searched for the text before the abstract
            content = file.read()
            log.write(f"{filename} is opened\n")
            res = re.search(r"((.|\n| )*?)(A[BSTRACTbstract]{7})", content)
            #regex finds the text in front of the abstract which should contain the title and the author informations

        if res: #if a match is found the collected string is seperated at linebreaks and turned into a list
            entry = res.group().split("\n")
            listen.write(str(entry)+"\n")
            if "abstract" in entry[-1].lower(): #if the last list entry contains abstract
                string_to_replace = ""          #every paper gets it own string that has to be replaced
                for x in range(1, len(entry)-1, 1): #the first entry, usually the title, is skipped
                    if x == (len(entry)-2) and entry[x] == "": #if the second to last entry is empty
                        string_to_replace += entry[x]           #then only that empty entry is added to the string
                        counter += 1
                        
                    else: 
                        string_to_replace += entry[x] + "\n" #every other entry gets added to the string with an additional \n to recreate the original string (before it was split) without the first element(usually the title) and the abstract header

            check = False
            already_used = os.listdir("Anon_Human_Papers") #collects filenames in list
            while check == False:       #loops until a random number is created that is not yet a filename in directory
                rand = random.randrange(0, len(liste), 1)   #ensures that the order of the files is randomized
                test = str(rand)+".md"                      #this way the order of files doesn't give an indication 
                                                            #of the source of the paper 
                if test not in already_used:
                    check = True

            anon_paper = content.replace(string_to_replace, "\n") #replaces the reconstructed string with a new line

            sub_dict = {}
            sub_dict["anon"] = filename
            sub_dict["Replaced String"] = string_to_replace
            key[rand] = sub_dict    #saves new filename, previous filename as well as the replaced string

            new = open(f"Anon_Human_Papers/{rand}.md", "a+", encoding="utf-8")
            new.write(anon_paper)       #writes anonymized paper in new file with new name
            new.close()

            strings.write(f"{string_to_replace}\n"+"*"*20+"\n\n")
            log.write(f"{filename} is finished:\n\n"+"*"*20+"\n\n")

        else:
            failure.append(filename)    #in case of error: filename is collected to check


    except Exception as e:      #in case of error: filename is collected to check
        log.write(f"Something is wrong with {filename}: {e}\n\n"+"*"*20+"\n\n")
        failure.append(filename)

df = pd.DataFrame.from_dict(key, orient='index')    #key dictionary transformed into pandas DataFrame and saved as
df = df.sort_index(axis = 0)                        #Excel-file for convenience
df.to_excel("Key_for_anon_paper.xlsx")

log.close()
strings.close()
listen.close()

To make it harder to differentiate between the generated and the human papers, the way figures are included within the text are homogenized or more specifically simplified

In [3]:
import pandas as pd
import os
import re
import random

liste = os.listdir("Anon_Human_Papers")


failure = []
counter = 0
for filename in liste:
    path = "Anon_Human_Papers/" + filename

    try:
        with open(path, "r+", encoding="utf-8") as file: 
            content = file.read()
            res = re.sub(r"\!\[\]\((.*?).jpeg", "![](Figure.jpeg",content)
            #regex finds and substitutes the previous text for figures and simplifies it

        if res: 
            counter+=1

            new = open(f"Anon_Human_Papers_Simplified_Figures/{filename}.md", "a+", encoding="utf-8")
            new.write(res)       #writes simplified paper 
            new.close()

        else:
            failure.append(filename)    #in case of error: filename is collected to check


    except Exception as e:      #in case of error: filename is collected to check
        log.write(f"Something is wrong with {filename}: {e}\n\n"+"*"*20+"\n\n")
        failure.append(filename)


To fix the names of the files as they curretnly have .md.md at the end and not just .md

In [2]:
import os

In [6]:
liste = os.listdir("Anon_Human_Papers_Simplified_Figures")
for file in liste:
    os.rename(f"Anon_Human_Papers_Simplified_Figures/{file}", f"Anon_Human_Papers_Simplified_Figures/{file[:-3]}")